# **Driver Advisory LLM — ZMQ Receiver + RAG**

**Pipeline:**
1. Receives 30-second aggregated payloads from `inference.ipynb` via ZMQ (port 5555)
2. Skips the LLM entirely if the driver is fully alert (all-zero payload)
3. Translates the payload into a natural-language narrative
4. **[RAG]** Selects the 2 most relevant knowledge chunks from `rag_knowledge_base.json`
5. **[RAG]** Injects those chunks into the prompt as grounding context
6. Feeds the enriched prompt into the local LLM (`llama-cpp-python`)
7. Outputs structured JSON: `{driver_state, risk_level, message}` — `message` is ready for TTS

**Token budget note:** Small models (SmolLM2-135M, TinyLlama) have a 512-token context.
RAG chunks average ~60-80 tokens each. This notebook injects a max of **2 chunks** to stay
safely within the window. If you switch to Mistral-7B (N_CTX=2048+), raise `RAG_TOP_K` to 4-5.

**Run order:** Start this notebook first, then run `inference.ipynb`.

## Cell 1 — Install Dependencies

In [8]:
import sys
!{sys.executable} -m pip install pyzmq llama-cpp-python --quiet


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip3.11 install --upgrade pip


## Cell 2 — Configuration

In [ ]:
# ZMQ CONFIG 
ZMQ_PORT = 5555                          # Must match inference.ipynb!!!

# MODEL CONFIG IF NEED
MODEL_PATH   = "Models/SmolLM2-135M-Instruct-Q4_K_M.gguf"   # fastest
# MODEL_PATH = "Models/TinyLlama-1.1B-Chat-v1.0.Q4_K_M.gguf"  # more coherent
# MODEL_PATH = "Models/mistral-7b-instruct-v0.1.Q4_K_M.gguf"   # best quality, ~11s

N_CTX        = 512    # Increase to 2048 if using Mistral-7B
MAX_TOKENS   = 80
TEMPERATURE  = 0.3

# RAG CONFIG
RAG_PATH  = "llm/rag_knowledge_base.json"   # Path to your knowledge base file
RAG_TOP_K = 2   # Max chunks to inject. Keep at 2 for small models (512 ctx).
                # Raise to 4-5 if using Mistral-7B with larger context window.

# ALERT THRESHOLD
SKIP_IF_ALL_CLEAR = True

## Cell 3 — Load the LLM Model

In [10]:
from llama_cpp import Llama

print(f"Loading model from: {MODEL_PATH}")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    verbose=False
)
print("Model loaded and ready.")

llama_context: n_ctx_seq (512) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


Loading model from: Models/SmolLM2-135M-Instruct-Q4_K_M.gguf
Model loaded and ready.


## Cell 4 — Load RAG Knowledge Base

In [12]:
import json

with open(RAG_PATH, "r") as f:
    rag_data = json.load(f)

# Store chunks in a flat list for easy lookup
CHUNKS = rag_data["chunks"]
print(f"Loaded {len(CHUNKS)} knowledge chunks from '{RAG_PATH}'")

# Preview the chunk types available
types = set(c["type"] for c in CHUNKS)
print(f"Chunk types: {types}")

Loaded 35 knowledge chunks from 'llm/rag_knowledge_base.json'
Chunk types: {'safety_guideline', 'mechanism', 'preventative_guideline', 'behavioral_guideline', 'short_term_intervention', 'risk_explanation'}


## Cell 5 — RAG Retrieval + Helper Functions

**How retrieval works (no vector DB needed):**
The knowledge base is small enough (35 chunks) that we use a **scored keyword match**.
Each chunk gets a relevance score based on how well its `tags` and `type` match
the current payload's conditions. The top-K chunks are returned.

**Scoring logic:**
- Microsleep events → boosts `high-risk`, `micro-sleep`, `stop-driving` chunks
- Drowsy events → boosts `short_term_intervention`, `nap`, `caffeine` chunks
- Worsening trend → boosts `risk_explanation` chunks
- Long microsleep (>5s) → adds extra weight to critical risk chunks

In [ ]:
import re


# RAG RETRIEVAL
def score_chunk(chunk: dict, payload: dict) -> int:
    """
    Returns a relevance score for a chunk given the current payload.
    Higher score = more relevant. Zero means not relevant.
    """
    ms    = payload.get("microsleep_events", 0)
    dr    = payload.get("drowsy_events", 0)
    lmf   = payload.get("longest_microsleep_frames", 0)
    trend = payload.get("drowsy_trend", 0)

    tags      = set(chunk.get("tags", []))
    chunk_type = chunk.get("type", "")
    score     = 0

    # Microsleep rules
    if ms > 0:
        if "high-risk" in tags:        score += 3
        if "micro-sleep" in tags:      score += 3
        if "stop-driving" in tags:     score += 2
        if chunk_type == "risk_explanation": score += 1

    # Long microsleep (>5s at 30fps = 150 frames)
    if lmf > 150:
        if "high-risk" in tags:        score += 2   # extra urgency
        if "stop-driving" in tags:     score += 2

    # Drowsy event rules
    if dr > 0:
        if chunk_type == "short_term_intervention": score += 3
        if "nap" in tags:              score += 2
        if "caffeine" in tags:         score += 1
        if "rest-stop" in tags:        score += 1

    # Worsening trend
    if trend > 50:
        if chunk_type == "risk_explanation":  score += 2
        if "sleep-deprivation" in tags: score += 1

    # Preventative content is lower priority during active events
    if chunk_type == "preventative_guideline" and (ms > 0 or dr > 0):
        score = max(0, score - 1)   # slight penalty — not the right moment

    return score


def retrieve_chunks(payload: dict, top_k: int = RAG_TOP_K) -> list:
    """
    Scores all chunks against the payload and returns the top_k most relevant.
    Chunks with score == 0 are excluded entirely.
    """
    scored = [(chunk, score_chunk(chunk, payload)) for chunk in CHUNKS]
    scored = [(c, s) for c, s in scored if s > 0]   # drop irrelevant
    scored.sort(key=lambda x: x[1], reverse=True)
    return [c for c, _ in scored[:top_k]]


def chunks_to_context(chunks: list) -> str:
    """
    Formats retrieved chunks into a concise context block for the prompt.
    Each chunk contributes its title + text only (no source URL) to save tokens.
    """
    if not chunks:
        return ""
    lines = []
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"[Ref {i}] {chunk['title']}: {chunk['text']}")
    return "\n".join(lines)


# PAYLOAD → NARRATIVE
def is_all_clear(payload: dict) -> bool:
    return (
        payload.get("microsleep_events", 0) == 0
        and payload.get("drowsy_events", 0) == 0
        and abs(payload.get("drowsy_trend", 0)) < 10
    )


def payload_to_narrative(payload: dict) -> str:
    ms    = payload.get("microsleep_events", 0)
    dr    = payload.get("drowsy_events", 0)
    lmf   = payload.get("longest_microsleep_frames", 0)
    ldf   = payload.get("longest_drowsy_frames", 0)
    trend = payload.get("drowsy_trend", 0)

    lmf_sec = round(lmf / 30, 1)
    ldf_sec = round(ldf / 30, 1)

    trend_desc = "stable"
    if trend > 50:   trend_desc = "rapidly worsening"
    elif trend > 10: trend_desc = "gradually worsening"
    elif trend < -50: trend_desc = "rapidly improving"
    elif trend < -10: trend_desc = "gradually improving"

    parts = ["In the last 30 seconds:"]
    if ms > 0:
        parts.append(f"The driver had {ms} microsleep episode(s), the longest lasting {lmf_sec}s.")
    if dr > 0:
        parts.append(f"The driver had {dr} drowsy episode(s), the longest lasting {ldf_sec}s.")
    if ms == 0 and dr == 0:
        parts.append("No microsleep or drowsy events were detected.")
    parts.append(f"Overall drowsiness trend: {trend_desc} (score: {trend}).")
    return " ".join(parts)


# PROMPT BUILDER (RAG-AWARE)
def build_prompt(narrative: str, context: str) -> str:
    """
    SmolLM2-135M optimized TTS prompt.
    Goal: single natural spoken sentence, no repetition of input text.
    """

    # Build compact input (avoid long RAG dumps triggering copy behavior)
    if context:
        user_msg = (
            "SAFETY NOTES (DO NOT REPEAT):\n"
            f"{context}\n\n"
            "DRIVING STATUS SUMMARY:\n"
            f"{narrative}\n\n"
            "TASK: Say ONE short safety instruction to the driver.\n"
            "RULES:\n"
            "- One sentence only\n"
            "- Do NOT repeat the status text\n"
            "- Do NOT mention numbers, refs, or sources\n"
            "- Do NOT explain\n"
            "- Speak like a calm voice assistant\n"
        )
    else:
        user_msg = (
            "DRIVING STATUS SUMMARY:\n"
            f"{narrative}\n\n"
            "TASK: Say ONE short safety instruction to the driver.\n"
            "RULES:\n"
            "- One sentence only\n"
            "- Do NOT repeat the status text\n"
            "- Do NOT mention numbers or sources\n"
            "- Do NOT explain\n"
            "- Speak like a calm voice assistant\n"
        )

    return (
        "<|im_start|>system\n"
        "You are a real-time driving safety voice assistant. "
        "You NEVER repeat user input. You ALWAYS rephrase in new words. "
        "You output ONLY one spoken sentence.<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

# LLM OUTPUT PARSER
def parse_llm_output(raw: str) -> dict:
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        match = re.search(r"\{.*?\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return {
        "driver_state": "Unknown",
        "risk_level":   "Unknown",
        "message":      "Unable to assess driver state."
    }


# FULL PIPELINE
def run_llm_with_rag(payload: dict) -> dict:
    """
    Full pipeline: payload → RAG retrieval → narrative → enriched prompt → LLM → result.
    """
    # 1. Retrieve relevant chunks
    chunks  = retrieve_chunks(payload)
    context = chunks_to_context(chunks)

    # 2. Build narrative + prompt
    narrative = payload_to_narrative(payload)
    prompt    = build_prompt(narrative, context)

    # 3. Run LLM
    response = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stop=["\n\n", "[INST]"]
    )

    raw_text = response["choices"][0]["text"]
    result   = parse_llm_output(raw_text)

    # Attach which chunks were used (useful for debugging)
    result["_rag_refs"] = [c["id"] for c in chunks]
    return result


print("RAG + helper functions ready.")

RAG + helper functions ready.


## Cell 6 — Main Loop (ZMQ Receiver + RAG + LLM)

Connects to port 5555 and processes each 30-second payload.
Stop with **Kernel → Interrupt**.

In [ ]:
import zmq
import time

# CONFIG
ZMQ_PORT = 5555
SKIP_IF_ALL_CLEAR = True

# DRIVER STATE LOGIC (PYTHON CONTROLLED)
def compute_severity(payload):
    microsleep = payload.get("microsleep_events", 0)
    drowsy = payload.get("drowsy_events", 0)
    trend = payload.get("drowsy_trend", 0)

    if microsleep >= 3 or trend > 150:
        return "CRITICAL"
    if microsleep >= 1 and drowsy >= 1:
        return "HIGH"
    if drowsy >= 2 or trend > 50:
        return "MODERATE"
    return "LOW"


def is_all_clear(payload):
    return (
        payload.get("microsleep_events", 0) == 0 and
        payload.get("drowsy_events", 0) == 0 and
        payload.get("drowsy_trend", 0) <= 5
    )

# PROMPT BUILDER (TTS ONLY)
def build_prompt(narrative: str, context: str, severity: str) -> str:
    if context:
        user_msg = (
            "SAFETY CONTEXT (DO NOT REPEAT):\n"
            f"{context}\n\n"
            f"DRIVING STATUS:\n{narrative}\n\n"
            f"SEVERITY: {severity}\n\n"
            "TASK: Say ONE short spoken instruction to the driver.\n"
            "RULES:\n"
            "- ONE sentence only\n"
            "- Do NOT repeat input\n"
            "- Do NOT mention numbers, refs, or sources\n"
            "- Do NOT explain\n"
            "- Speak naturally like a voice assistant\n"
        )
    else:
        user_msg = (
            f"DRIVING STATUS:\n{narrative}\n\n"
            f"SEVERITY: {severity}\n\n"
            "TASK: Say ONE short spoken instruction to the driver.\n"
            "RULES:\n"
            "- ONE sentence only\n"
            "- Do NOT repeat input\n"
            "- Do NOT mention numbers or sources\n"
            "- Do NOT explain\n"
            "- Speak naturally like a voice assistant\n"
        )

    return (
        "<|im_start|>system\n"
        "You are a real-time vehicle safety voice assistant. "
        "You NEVER repeat user input. You ALWAYS paraphrase. "
        "You output ONLY one sentence for speech.\n"
        "<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

# MAIN LOOP
context = zmq.Context()
socket  = context.socket(zmq.PULL)
socket.connect(f"tcp://localhost:{ZMQ_PORT}")

print(f"LLM Advisor (TTS mode) listening on port {ZMQ_PORT}...")
print("Waiting for payloads...\n")

try:
    while True:
        payload = socket.recv_json()
        print(f"[RECEIVED] {payload}")

        # Skip clean state
        if SKIP_IF_ALL_CLEAR and is_all_clear(payload):
            print("[SKIP] Driver is fully alert\n")
            continue

        # =========================
        # RAG STEP
        # =========================
        chunks = retrieve_chunks(payload)
        context_text = chunks_to_context(chunks) if chunks else ""
        narrative = payload_to_narrative(payload)

        severity = compute_severity(payload)

        # =========================
        # LLM CALL
        # =========================
        prompt = build_prompt(narrative, context_text, severity)

        t0 = time.time()
        response = llm(
            prompt,
            max_tokens=40,
            temperature=0.4,
            stop=["\n\n"]
        )
        elapsed = time.time() - t0

        message = response["choices"][0]["text"].strip()

        # =========================
        # OUTPUT
        # =========================
        print(f"[LLM] ({elapsed:.2f}s) severity={severity}")
        print(f"[TTS READY] → {message}")

        rag_refs = [c["id"] for c in chunks] if chunks else []
        print(f"[REFS USED] {rag_refs}\n")

        # Optional TTS hook
        # tts_engine.say(message)

except KeyboardInterrupt:
    print("\nStopped by user.")

finally:
    socket.close()
    context.term()
    print("ZMQ connection closed.")

LLM Advisor (TTS mode) listening on port 5555...
Waiting for payloads...

[RECEIVED] {'microsleep_events': 4, 'drowsy_events': 7, 'longest_microsleep_frames': 61, 'longest_drowsy_frames': 236, 'drowsy_trend': -100}
[LLM] (1.60s) severity=CRITICAL
[TTS READY] → SAFETY CONTEXT:
[Ref 1] Caffeine + 20-minute nap strategy: If you become sleepy while driving, consume one to two cups of coffee and pull over
[REFS USED] ['nhtsa_intervention_002', 'allstate_006']

[RECEIVED] {'microsleep_events': 1, 'drowsy_events': 0, 'longest_microsleep_frames': 19, 'longest_drowsy_frames': 0, 'drowsy_trend': 1}
[LLM] (1.76s) severity=LOW
[TTS READY] → SAFETY CONTEXT:
[Ref 1] Micro-sleeps and crash risk: Caffeine alone may not prevent drowsy driving. Severely sleep-deprived drivers may experience
[REFS USED] ['nhtsa_micro_sleep_001', 'nhtsa_intervention_002']

[RECEIVED] {'microsleep_events': 0, 'drowsy_events': 0, 'longest_microsleep_frames': 0, 'longest_drowsy_frames': 0, 'drowsy_trend': 0}
[SKIP] Driver is

## Cell 7 — Debug: See Raw LLM Output <- Cell 7 and 8 Only for Debuging!!!
Run this first to see exactly what the model is generating before JSON parsing.

In [29]:
def build_prompt(narrative: str, context: str) -> str:
    """
    SmolLM2-135M optimized TTS prompt.
    Goal: single natural spoken sentence, no repetition of input text.
    """

    # Build compact input (avoid long RAG dumps triggering copy behavior)
    if context:
        user_msg = (
            "SAFETY NOTES (DO NOT REPEAT):\n"
            f"{context}\n\n"
            "DRIVING STATUS SUMMARY:\n"
            f"{narrative}\n\n"
            "TASK: Say ONE short safety instruction to the driver.\n"
            "RULES:\n"
            "- One sentence only\n"
            "- Do NOT repeat the status text\n"
            "- Do NOT mention numbers, refs, or sources\n"
            "- Do NOT explain\n"
            "- Speak like a calm voice assistant\n"
        )
    else:
        user_msg = (
            "DRIVING STATUS SUMMARY:\n"
            f"{narrative}\n\n"
            "TASK: Say ONE short safety instruction to the driver.\n"
            "RULES:\n"
            "- One sentence only\n"
            "- Do NOT repeat the status text\n"
            "- Do NOT mention numbers or sources\n"
            "- Do NOT explain\n"
            "- Speak like a calm voice assistant\n"
        )

    return (
        "<|im_start|>system\n"
        "You are a real-time driving safety voice assistant. "
        "You NEVER repeat user input. You ALWAYS rephrase in new words. "
        "You output ONLY one spoken sentence.<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

In [30]:
# Debug: print the raw text the model produces so we can see what's going wrong
test_payload = {
    'microsleep_events': 2, 'drowsy_events': 3,
    'longest_microsleep_frames': 45, 'longest_drowsy_frames': 44,
    'drowsy_trend': 24
}

chunks    = retrieve_chunks(test_payload)
context   = chunks_to_context(chunks)
narrative = payload_to_narrative(test_payload)
prompt    = build_prompt(narrative, context)

print("=== PROMPT SENT TO MODEL ===")
print(prompt)
print()

response = llm(
    prompt,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    stop=["\n\n", "[INST]"]
)

raw = response["choices"][0]["text"]
print("=== RAW MODEL OUTPUT ===")
print(repr(raw))   # repr shows hidden characters like \n
print()
print(raw)

=== PROMPT SENT TO MODEL ===
<|im_start|>system
You are a real-time driving safety voice assistant. You NEVER repeat user input. You ALWAYS rephrase in new words. You output ONLY one spoken sentence.<|im_end|>
<|im_start|>user
SAFETY NOTES (DO NOT REPEAT):
[Ref 1] Caffeine + 20-minute nap strategy: If you become sleepy while driving, consume one to two cups of coffee and pull over for a 20-minute nap in a safe, well-lit rest area. This combination temporarily improves alertness but is not a substitute for sleep.
[Ref 2] Combined nap and caffeine approach: If drowsiness occurs while driving, pull over safely, drink one to two cups of coffee, and take a brief nap. Caffeine may take around 30 minutes to take effect, so rest is essential.

DRIVING STATUS SUMMARY:
In the last 30 seconds: The driver had 2 microsleep episode(s), the longest lasting 1.5s. The driver had 3 drowsy episode(s), the longest lasting 1.5s. Overall drowsiness trend: gradually worsening (score: 24).

TASK: Say ONE shor

In [31]:
# Test payloads
test_payloads = [
    {
        # Critical: microsleep + drowsy + worsening trend
        'microsleep_events': 2,
        'drowsy_events': 3,
        'longest_microsleep_frames': 45,
        'longest_drowsy_frames': 44,
        'drowsy_trend': 24
    },

    {
        # Severe: very long microsleep episode (>13s)
        'microsleep_events': 1,
        'drowsy_events': 1,
        'longest_microsleep_frames': 398,
        'longest_drowsy_frames': 76,
        'drowsy_trend': -80
    },

    {
        # Drowsy only, rapidly worsening trend
        'microsleep_events': 0,
        'drowsy_events': 3,
        'longest_microsleep_frames': 0,
        'longest_drowsy_frames': 185,
        'drowsy_trend': 200
    },

    {
        # All clear — should be skipped
        'microsleep_events': 0,
        'drowsy_events': 0,
        'longest_microsleep_frames': 0,
        'longest_drowsy_frames': 0,
        'drowsy_trend': 0
    },
]


def debug_model_outputs(payloads):
    for i, test_payload in enumerate(payloads, start=1):

        print("=" * 80)
        print(f"TEST PAYLOAD #{i}")
        print("=" * 80)
        print(test_payload)
        print()

        chunks = retrieve_chunks(test_payload)
        context = chunks_to_context(chunks)
        narrative = payload_to_narrative(test_payload)
        prompt = build_prompt(narrative, context)

        print("=== PROMPT SENT TO MODEL ===")
        print(prompt)
        print()

        response = llm(
            prompt,
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            stop=["\n\n", "[INST]"]
        )

        raw = response["choices"][0]["text"]

        print("=== RAW MODEL OUTPUT ===")
        print(repr(raw))  # repr shows hidden characters like \n
        print()
        print(raw)
        print("\n\n")


# Run all tests
debug_model_outputs(test_payloads)

TEST PAYLOAD #1
{'microsleep_events': 2, 'drowsy_events': 3, 'longest_microsleep_frames': 45, 'longest_drowsy_frames': 44, 'drowsy_trend': 24}

=== PROMPT SENT TO MODEL ===
<|im_start|>system
You are a real-time driving safety voice assistant. You NEVER repeat user input. You ALWAYS rephrase in new words. You output ONLY one spoken sentence.<|im_end|>
<|im_start|>user
SAFETY NOTES (DO NOT REPEAT):
[Ref 1] Caffeine + 20-minute nap strategy: If you become sleepy while driving, consume one to two cups of coffee and pull over for a 20-minute nap in a safe, well-lit rest area. This combination temporarily improves alertness but is not a substitute for sleep.
[Ref 2] Combined nap and caffeine approach: If drowsiness occurs while driving, pull over safely, drink one to two cups of coffee, and take a brief nap. Caffeine may take around 30 minutes to take effect, so rest is essential.

DRIVING STATUS SUMMARY:
In the last 30 seconds: The driver had 2 microsleep episode(s), the longest lasting 1.

## Cell 8 — (Optional) Quick Test Without ZMQ

Verify RAG retrieval and LLM output without needing the full camera pipeline.
Also prints which knowledge chunks were selected for each scenario.

In [ ]:
import time

test_payloads = [
    # Critical: microsleep + drowsy + worsening trend
    {'microsleep_events': 2, 'drowsy_events': 3,
     'longest_microsleep_frames': 45, 'longest_drowsy_frames': 44,
     'drowsy_trend': 24},

    # Severe: very long microsleep episode (>13s)
    {'microsleep_events': 1, 'drowsy_events': 1,
     'longest_microsleep_frames': 398, 'longest_drowsy_frames': 76,
     'drowsy_trend': -80},

    # Drowsy only, rapidly worsening trend
    {'microsleep_events': 0, 'drowsy_events': 3,
     'longest_microsleep_frames': 0, 'longest_drowsy_frames': 185,
     'drowsy_trend': 200},

    # All clear — should be skipped
    {'microsleep_events': 0, 'drowsy_events': 0,
     'longest_microsleep_frames': 0, 'longest_drowsy_frames': 0,
     'drowsy_trend': 0},
]

for i, payload in enumerate(test_payloads):
    print(f"{'='*60}")
    print(f"Test {i+1}: {payload}")

    if SKIP_IF_ALL_CLEAR and is_all_clear(payload):
        print("→ [SKIPPED — driver is alert]\n")
        continue

    # Show RAG selection
    chunks = retrieve_chunks(payload)
    print(f"RAG chunks selected ({len(chunks)}):")
    for c in chunks:
        print(f"  • [{c['id']}] {c['title']}")

    # Run LLM
    t0     = time.time()
    result = run_llm_with_rag(payload)
    refs   = result.pop("_rag_refs", [])
    print(f"LLM result ({time.time()-t0:.2f}s): {result}")
    print(f"TTS → {result.get('message', '')}")
    print()